# Track 1, Option A — Notebook 02: Preference Fine-Tuning (DPO, 5 LoRA trials)

Starts from the **best SFT model** (Notebook 01), merges its LoRA into the base, then
runs **5 DPO trials** on `HuggingFaceH4/ultrafeedback_binarized` varying **beta, LR,
batch size, epochs**. Evaluates each on the 10-prompt test set (BLEU + BERTScore) and
auto-selects the best SFT+DPO model.

**Before running**, make the best SFT model available in one of two ways:
1. **Reuse adapter (preferred):** add Notebook 01's output as an input dataset so
   `best_sft_adapter/` is on disk — it is detected automatically; **or**
2. **Retrain inline:** set `BEST_SFT_CONFIG` below to the winning config printed by
   Notebook 01, and it will be retrained before DPO.

**Kaggle setup:** GPU (T4 x2 or P100), Internet **ON**.

**Outputs (for the report):**
- `dpo_results.csv` — per-trial summary (BLEU/chrF/ROUGE-L/BERTScore, DPO reward accuracy + margin, per-difficulty breakdown, trainable params, GPU memory, time)
- `dpo_per_prompt.csv` — per-prompt scores for every trial
- `dpo_training_log.csv` — loss + reward curves; `dpo_by_difficulty.csv` — breakdown
- `dpo_preprocessing_stats.json` — preference-margin filtering stats
- `best_dpo_config.json`, `dpo_sample_outputs.md`, `dpo_showcase_outputs.md`

In [2]:
%%capture
!pip install -q -U "transformers>=4.51.0" "trl>=0.15.0" "peft>=0.14.0" \
    "datasets>=3.2.0" "accelerate>=1.3.0" "evaluate>=0.4.3" \
    "sacrebleu>=2.4.3" "bert-score>=0.3.13" "sentencepiece>=0.2.0" \
    "rouge-score>=0.1.2"
# Kaggle ships an old torchao (0.10) that PEFT's LoRA dispatcher rejects, even
# though we never use it. Remove it to avoid the ImportError in get_peft_model().
!pip uninstall -y torchao

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"             # train on ONE GPU (T4 x2 'auto' sharding stalls)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduce GPU fragmentation
import json, gc, glob, random, time, shutil, numpy as np, pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig, SFTTrainer, SFTConfig
import sacrebleu
from bert_score import score as _bertscore
from tqdm.auto import tqdm

# ---------------- Config ----------------
BASE_MODEL     = "Qwen/Qwen3-0.6B-Base"
SYSTEM_PROMPT  = "You are a helpful assistant. Answer concisely and directly."
MAX_NEW_TOKENS = 256
MAX_LEN        = 512    # capped for T4 memory (DPO scores chosen+rejected = 2x logits)
MAX_PROMPT_LEN = 256
N_TRAIN        = 2000
N_VAL          = 300
SEED           = 42
OUT_DIR        = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
MERGED_DIR     = os.path.join(OUT_DIR, "sft_merged")

# If best_sft_adapter/ is NOT found on disk, this config is retrained as the SFT
# starting point. Copy the winning config from Notebook 01's output here.
BEST_SFT_CONFIG = {"name": "sft2", "r": 16, "alpha": 32,
                   "targets": ["q_proj", "k_proj", "v_proj", "o_proj"],
                   "lr": 2e-4, "bs": 2, "ga": 4, "epochs": 2}

set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
# T4 (Turing) / P100 (Pascal) have fp16 tensor cores but NO native bf16 -> bf16 is
# emulated and ~2x slower. Force fp16 so matmuls hit the tensor cores.
DTYPE  = torch.float16
USE_BF16 = False
print("dtype:", DTYPE)

CHATML_TEMPLATE = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

dtype: torch.float16


In [4]:
# ---------------- Test set + shared eval utilities ----------------
INLINE_TEST_SET = {"prompts": [
    {"id": 1, "type": "factual_short", "difficulty": "easy", "prompt": "Which planet is known as the Red Planet? Answer in one short sentence.", "gold": "Mars is known as the Red Planet."},
    {"id": 2, "type": "sentiment_classification", "difficulty": "easy", "prompt": "Classify the sentiment of this sentence as positive or negative: 'I absolutely love this movie.'", "gold": "The sentiment is positive."},
    {"id": 3, "type": "math_reasoning", "difficulty": "easy", "prompt": "A train travels 60 km in 1.5 hours. What is its average speed in km/h?", "gold": "The average speed is 40 km/h."},
    {"id": 4, "type": "list_constrained", "difficulty": "medium", "prompt": "List exactly three primary colors, separated by commas.", "gold": "Red, yellow, blue."},
    {"id": 5, "type": "rewriting_tone", "difficulty": "medium", "prompt": "Rewrite this sentence to be more formal: 'Hey, can you send me that file ASAP?'", "gold": "Could you please send me that file at your earliest convenience?"},
    {"id": 6, "type": "summarization_grounded", "difficulty": "medium", "prompt": "Summarize the following text in one sentence: 'The library will be closed on Monday for a public holiday. It will reopen on Tuesday at 9 a.m. with normal hours.'", "gold": "The library is closed Monday for a public holiday and reopens Tuesday at 9 a.m."},
    {"id": 7, "type": "extraction", "difficulty": "medium", "prompt": "Extract when the library reopens from this text: 'The library will be closed on Monday for a public holiday. It will reopen on Tuesday at 9 a.m. with normal hours.'", "gold": "Tuesday at 9 a.m."},
    {"id": 8, "type": "sentiment_3class", "difficulty": "medium", "prompt": "Classify the following review as positive, negative, or neutral: 'The food was okay, nothing special.'", "gold": "The sentiment is neutral."},
    {"id": 9, "type": "unit_conversion", "difficulty": "medium", "prompt": "Convert 2 kilometers to meters.", "gold": "2 kilometers is 2000 meters."},
    {"id": 10, "type": "coding_oneliner", "difficulty": "medium", "prompt": "Write a single line of Python that computes the sum of a list named nums.", "gold": "total = sum(nums)"},
    {"id": 11, "type": "grammar", "difficulty": "medium", "prompt": "Give the past tense of the verb 'run' in a single word.", "gold": "Ran."},
    {"id": 12, "type": "translation", "difficulty": "medium", "prompt": "Translate 'Good morning' into French.", "gold": "Bonjour."},
    {"id": 13, "type": "date_reasoning", "difficulty": "harder", "prompt": "If today is Monday, what day will it be in 3 days?", "gold": "It will be Thursday."},
    {"id": 14, "type": "multi_step_math", "difficulty": "harder", "prompt": "A pizza is cut into 8 slices. If 3 people each eat 2 slices, how many slices remain?", "gold": "Two slices remain."},
    {"id": 15, "type": "explanation_audience", "difficulty": "harder", "prompt": "Explain what a CPU does in exactly two sentences.", "gold": "A CPU carries out the instructions of a program. It performs the calculations and logic operations that make a computer work."},
]}

def load_test_set():
    for p in ["test_set.json", "/kaggle/input/test-set/test_set.json", "/kaggle/working/test_set.json"]:
        if os.path.exists(p):
            with open(p) as f: return json.load(f)
    return INLINE_TEST_SET

PROMPTS = load_test_set()["prompts"]

def load_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.chat_template = CHATML_TEMPLATE
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    return tok

def build_prompt(tokenizer, user_msg):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def _eos_ids(tok):
    ids = []
    im_end = tok.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end >= 0: ids.append(im_end)
    if tok.eos_token_id is not None: ids.append(tok.eos_token_id)
    return ids or None

@torch.no_grad()
def generate_response(model, tokenizer, user_msg, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_prompt(tokenizer, user_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id, eos_token_id=_eos_ids(tokenizer))
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

from rouge_score import rouge_scorer
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def _bertscore_prf(preds, refs):
    P, R, F1 = _bertscore(preds, refs, lang="en", rescale_with_baseline=True)
    return ([float(x) * 100 for x in P], [float(x) * 100 for x in R], [float(x) * 100 for x in F1])

def evaluate_model(model, tokenizer, prompts, tag="model"):
    """Generate on every prompt and score with a comprehensive metric suite.
    Returns a per-prompt DataFrame (one row per prompt, all metrics)."""
    preds, refs, rows = [], [], []
    for p in tqdm(prompts, desc=f'generate [{tag}]'):
        pred = generate_response(model, tokenizer, p["prompt"])
        preds.append(pred); refs.append(p["gold"])
        rows.append({"trial": tag, "id": p["id"], "type": p["type"],
                     "difficulty": p.get("difficulty", "NA"),
                     "prompt": p["prompt"], "gold": p["gold"], "prediction": pred,
                     "pred_words": len(pred.split()), "pred_chars": len(pred),
                     "gold_words": len(p["gold"].split())})
    bleus  = [sacrebleu.sentence_bleu(pr, [rf]).score for pr, rf in zip(preds, refs)]
    chrfs  = [sacrebleu.sentence_chrf(pr, [rf]).score for pr, rf in zip(preds, refs)]
    rouges = [_rouge.score(rf, pr)["rougeL"].fmeasure * 100 for pr, rf in zip(preds, refs)]
    bp, br, bf = _bertscore_prf(preds, refs)
    for i, r in enumerate(rows):
        r.update(bleu=round(bleus[i], 3), chrf=round(chrfs[i], 3), rougeL=round(rouges[i], 3),
                 bertscore_p=round(bp[i], 3), bertscore_r=round(br[i], 3), bertscore_f1=round(bf[i], 3))
    df = pd.DataFrame(rows)
    print(f"[{tag}] BLEU={df.bleu.mean():.2f} | chrF={df.chrf.mean():.2f} | "
          f"ROUGE-L={df.rougeL.mean():.2f} | BERT-F1={df.bertscore_f1.mean():.2f} | "
          f"avg_len={df.pred_words.mean():.0f}w")
    return df

def _tier(pp, difficulty, metric):
    sub = pp[pp["difficulty"] == difficulty]
    return round(float(sub[metric].mean()), 3) if len(sub) else float("nan")

tokenizer = load_tokenizer(BASE_MODEL)



print("done")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

done


In [5]:
# ---------------- Get the SFT starting point: load adapter, else retrain ----------------
def find_sft_adapter():
    cands = (["/kaggle/input/datasets/rockybhai159/puhasafd/best_sft_adapter"]  # attached 01 output
             + glob.glob("/kaggle/input/**/best_sft_adapter", recursive=True)
             + ["/kaggle/working/best_sft_adapter", "best_sft_adapter"])
    for p in cands:
        if os.path.isdir(p) and os.path.exists(os.path.join(p, "adapter_config.json")):
            return p
    return None

def retrain_best_sft():
    """Fallback: train BEST_SFT_CONFIG on ultrachat, return adapter dir."""
    cfg = BEST_SFT_CONFIG
    print("No saved adapter found -> retraining best SFT config:", cfg["name"])
    raw = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
    raw = raw.shuffle(seed=SEED).select(range(min(N_TRAIN * 2, len(raw))))
    def to_text(ex):
        m = ex["messages"]
        if not m or m[0]["role"] != "user" or len(m) < 2: return {"text": ""}
        turn = [{"role": "system", "content": SYSTEM_PROMPT}, m[0], m[1]]
        return {"text": tokenizer.apply_chat_template(turn, tokenize=False, add_generation_prompt=False)}
    ds = raw.map(to_text, remove_columns=raw.column_names).filter(lambda x: len(x["text"]) > 0).select(range(N_TRAIN))
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE).to("cuda")
    model.config.use_cache = False
    model.enable_input_require_grads()
    kw = dict(output_dir=os.path.join(OUT_DIR, "sft_retrain"),
              per_device_train_batch_size=min(cfg["bs"], 2), gradient_accumulation_steps=cfg["ga"],
              learning_rate=cfg["lr"], num_train_epochs=cfg["epochs"], logging_steps=25,
              save_strategy="no", warmup_ratio=0.03, lr_scheduler_type="cosine",
              gradient_checkpointing=False,
              bf16=USE_BF16, fp16=not USE_BF16, report_to="none", seed=SEED,
              dataset_text_field="text", packing=False)
    try:    args = SFTConfig(max_seq_length=MAX_LEN, **kw)
    except TypeError: args = SFTConfig(max_length=MAX_LEN, **kw)
    lora = LoraConfig(r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=0.05, bias="none",
                      task_type="CAUSAL_LM", target_modules=cfg["targets"])
    tr = SFTTrainer(model=model, args=args, train_dataset=ds,
                    processing_class=tokenizer, peft_config=lora)
    tr.train()
    out = os.path.join(OUT_DIR, "best_sft_adapter")
    tr.save_model(out)
    del tr, model; gc.collect(); torch.cuda.empty_cache()
    return out

adapter_dir = find_sft_adapter() or retrain_best_sft()
print("Using SFT adapter:", adapter_dir)

# Merge LoRA into the base -> this merged model is the DPO starting policy.
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE).to("cuda")
merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
if os.path.exists(MERGED_DIR): shutil.rmtree(MERGED_DIR)
merged.save_pretrained(MERGED_DIR); tokenizer.save_pretrained(MERGED_DIR)
del base, merged; gc.collect(); torch.cuda.empty_cache()
print("Merged SFT model saved ->", MERGED_DIR)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Using SFT adapter: /kaggle/input/datasets/rockybhai159/puhasafd/best_sft_adapter


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged SFT model saved -> /kaggle/working/sft_merged


In [6]:
# ---------------- Load + technically preprocess the DPO dataset (ultrafeedback_binarized) ---
# Preprocessing pipeline (each step logged for the report):
#   1. random subsample (fixed seed)          5. drop identical chosen==rejected
#   2. extract prompt + last-turn responses    6. preference-margin filter (score_chosen - score_rejected >= MARGIN_MIN)
#   3. whitespace normalization                7. dedup by prompt
#   4. drop too-short completions              8. render to ChatML prompt/chosen/rejected + record stats
import re
from datasets import Dataset

MIN_COMP_CHARS = 20    # drop near-empty completions
MARGIN_MIN     = 0.5   # keep only clearly-preferred pairs (UltraFeedback scores are ~1-10)

dpo_train_raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs").shuffle(seed=SEED).select(range(min(N_TRAIN * 3, 60000)))
dpo_val_raw   = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_prefs").shuffle(seed=SEED).select(range(min(N_VAL * 3, 6000)))

def _norm(s):
    return re.sub(r"\s+", " ", s or "").strip()

def build_prefs(ds, n_target):
    prompts, chosens, rejecteds, margins, seen = [], [], [], [], set()
    c = dict(raw=len(ds), too_short=0, identical=0, low_margin=0, duplicates=0)
    for ex in ds:
        user = _norm(ex.get("prompt", ""))
        ch, rj = ex["chosen"], ex["rejected"]
        chosen   = _norm(ch[-1]["content"] if isinstance(ch, list) else ch)
        rejected = _norm(rj[-1]["content"] if isinstance(rj, list) else rj)
        sc, sr = ex.get("score_chosen"), ex.get("score_rejected")
        if not user or len(chosen) < MIN_COMP_CHARS or len(rejected) < MIN_COMP_CHARS:
            c["too_short"] += 1; continue
        if chosen.lower() == rejected.lower():
            c["identical"] += 1; continue
        if sc is not None and sr is not None and (float(sc) - float(sr)) < MARGIN_MIN:
            c["low_margin"] += 1; continue
        key = user[:200].lower()
        if key in seen:
            c["duplicates"] += 1; continue
        seen.add(key)
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text); chosens.append(chosen + "<|im_end|>")
        rejecteds.append(rejected + "<|im_end|>")
        margins.append(float(sc) - float(sr) if sc is not None and sr is not None else float("nan"))
        if len(prompts) >= n_target:
            break
    c["kept"] = len(prompts)
    return prompts, chosens, rejecteds, margins, c

tr_p, tr_c, tr_r, tr_m, tr_stats = build_prefs(dpo_train_raw, N_TRAIN)
va_p, va_c, va_r, va_m, va_stats = build_prefs(dpo_val_raw, N_VAL)
dpo_train = Dataset.from_dict({"prompt": tr_p, "chosen": tr_c, "rejected": tr_r})
dpo_val   = Dataset.from_dict({"prompt": va_p, "chosen": va_c, "rejected": va_r})

_valid_m = [m for m in tr_m if m == m]  # drop NaN
prep_stats = {"dataset": "HuggingFaceH4/ultrafeedback_binarized",
              "train_filtering": tr_stats, "val_filtering": va_stats,
              "preference_margin": {"min_kept": MARGIN_MIN,
                                    "mean": round(float(np.mean(_valid_m)), 3) if _valid_m else None,
                                    "median": round(float(np.median(_valid_m)), 3) if _valid_m else None},
              "filters": {"min_completion_chars": MIN_COMP_CHARS, "dedup": "by lowercased prompt prefix",
                          "normalization": "collapse whitespace", "drop_identical_pairs": True}}
with open(os.path.join(OUT_DIR, "dpo_preprocessing_stats.json"), "w") as f:
    json.dump(prep_stats, f, indent=2)
print(json.dumps(prep_stats, indent=2))
print("\nDPO train:", len(dpo_train), "| val:", len(dpo_val))
print("---- example prompt ----\n", dpo_train[0]["prompt"][:400])
print("---- chosen ----\n", dpo_train[0]["chosen"][:200])

README.md: 0.00B [00:00, ?B/s]

data/train_prefs-00000-of-00001.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

data/test_prefs-00000-of-00001.parquet:   0%|          | 0.00/7.29M [00:00<?, ?B/s]

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/3.72M [00:00<?, ?B/s]

data/train_gen-00000-of-00001.parquet:   0%|          | 0.00/184M [00:00<?, ?B/s]

...

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

{
  "dataset": "HuggingFaceH4/ultrafeedback_binarized",
  "train_filtering": {
    "raw": 6000,
    "too_short": 176,
    "identical": 3,
    "low_margin": 257,
    "duplicates": 3,
    "kept": 2000
  },
  "val_filtering": {
    "raw": 900,
    "too_short": 37,
    "identical": 0,
    "low_margin": 49,
    "duplicates": 0,
    "kept": 300
  },
  "preference_margin": {
    "min_kept": 0.5,
    "mean": 2.055,
    "median": 1.5
  },
  "filters": {
    "min_completion_chars": 20,
    "dedup": "by lowercased prompt prefix",
    "normalization": "collapse whitespace",
    "drop_identical_pairs": true
  }
}

DPO train: 2000 | val: 300
---- example prompt ----
 <|im_start|>system
You are a helpful assistant. Answer concisely and directly.<|im_end|>
<|im_start|>user
Do you know something about crystallography and structure factor?<|im_end|>
<|im_start|>assistant

---- chosen ----
 Crystallography is the science of the arrangement of atoms in solids. It is a vast and interdisciplinary field that

In [7]:
# ---------------- 5 DPO trials: vary beta / LR / batch / epochs ----------------
# bs = per-device batch (kept at 1 for T4; DPO scores chosen+rejected = 2x logits);
# ga = grad-accum so the EFFECTIVE batch (bs*ga) still varies: 4, 8, 8, 16, 16.
DPO_TRIALS = [
    {"name": "dpo1", "beta": 0.10, "lr": 5e-5, "bs": 1, "ga": 4,  "epochs": 1},
    {"name": "dpo2", "beta": 0.10, "lr": 1e-5, "bs": 1, "ga": 8,  "epochs": 1},
    {"name": "dpo3", "beta": 0.30, "lr": 5e-6, "bs": 1, "ga": 8,  "epochs": 2},
    {"name": "dpo4", "beta": 0.05, "lr": 5e-5, "bs": 1, "ga": 16, "epochs": 1},
    {"name": "dpo5", "beta": 0.50, "lr": 1e-6, "bs": 1, "ga": 16, "epochs": 2},
]
# LoRA is held fixed across DPO trials (the assignment varies beta/LR/batch/epochs).
DPO_LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

def make_dpo_config(cfg, out):
    kw = dict(output_dir=out, per_device_train_batch_size=cfg["bs"],
              per_device_eval_batch_size=1, gradient_accumulation_steps=cfg["ga"],
              learning_rate=cfg["lr"], num_train_epochs=cfg["epochs"], logging_steps=25,
              eval_strategy="epoch", save_strategy="no", warmup_ratio=0.05,
              lr_scheduler_type="cosine", bf16=USE_BF16, fp16=not USE_BF16,
              gradient_checkpointing=False,
              report_to="none", seed=SEED, beta=cfg["beta"],
              max_length=MAX_LEN, max_prompt_length=MAX_PROMPT_LEN,
              remove_unused_columns=False)
    # TRL renamed/dropped some length args across versions -> keep only valid fields.
    from dataclasses import fields
    valid = {f.name for f in fields(DPOConfig)}
    dropped = [k for k in kw if k not in valid]
    if dropped:
        print("[make_dpo_config] dropping unsupported args:", dropped)
    return DPOConfig(**{k: v for k, v in kw.items() if k in valid})

def run_dpo_trial(cfg):
    set_seed(SEED)
    gc.collect(); torch.cuda.empty_cache()  # free any leftover memory before loading
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    out = os.path.join(OUT_DIR, cfg["name"])
    model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=DTYPE).to("cuda")
    model.config.use_cache = False
    model.enable_input_require_grads()  # required for gradient checkpointing + PEFT
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                      task_type="CAUSAL_LM", target_modules=DPO_LORA_TARGETS)
    t0 = time.time()
    trainer = DPOTrainer(model=model, ref_model=None, args=make_dpo_config(cfg, out),
                         train_dataset=dpo_train, eval_dataset=dpo_val,
                         processing_class=tokenizer, peft_config=lora)
    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in trainer.model.parameters())
    trainer.train()
    ev = trainer.evaluate()  # DPO eval includes reward accuracy / margins
    val_loss      = float(ev.get("eval_loss", float("nan")))
    reward_acc    = float(ev.get("eval_rewards/accuracies", float("nan")))
    reward_margin = float(ev.get("eval_rewards/margins", float("nan")))
    train_time = time.time() - t0
    peak_gb = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
    train_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    final_train_loss = float(train_losses[-1]) if train_losses else float("nan")
    log_df = pd.DataFrame(trainer.state.log_history); log_df.insert(0, "trial", cfg["name"])

    trainer.model.config.use_cache = True
    pp = evaluate_model(trainer.model, tokenizer, PROMPTS, tag=cfg["name"])
    trainer.save_model(out)

    res = {"name": cfg["name"], "beta": cfg["beta"], "lr": cfg["lr"],
           "per_device_bs": cfg["bs"], "grad_accum": cfg["ga"], "eff_batch": cfg["bs"] * cfg["ga"],
           "epochs": cfg["epochs"], "trainable_params": trainable,
           "trainable_pct": round(100 * trainable / total, 4),
           "final_train_loss": round(final_train_loss, 4), "eval_loss": round(val_loss, 4),
           "reward_accuracy": round(reward_acc, 4), "reward_margin": round(reward_margin, 4),
           "bleu": round(float(pp.bleu.mean()), 3), "chrf": round(float(pp.chrf.mean()), 3),
           "rougeL": round(float(pp.rougeL.mean()), 3),
           "bertscore_p": round(float(pp.bertscore_p.mean()), 3),
           "bertscore_r": round(float(pp.bertscore_r.mean()), 3),
           "bertscore_f1": round(float(pp.bertscore_f1.mean()), 3),
           "combined": round(float(pp.bleu.mean() + pp.bertscore_f1.mean()), 3),
           "bleu_easy": _tier(pp, "easy", "bleu"), "bleu_medium": _tier(pp, "medium", "bleu"),
           "bleu_harder": _tier(pp, "harder", "bleu"),
           "bertf1_easy": _tier(pp, "easy", "bertscore_f1"), "bertf1_medium": _tier(pp, "medium", "bertscore_f1"),
           "bertf1_harder": _tier(pp, "harder", "bertscore_f1"),
           "avg_pred_words": round(float(pp.pred_words.mean()), 1),
           "peak_gpu_gb": round(peak_gb, 2), "train_time_s": round(train_time, 1),
           "adapter_dir": out}
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return res, pp, log_df



print("done")

done


In [8]:
# ---------------- Run all 5 DPO trials ----------------
dpo_results, per_prompt_all, logs_all = [], [], []
for i, cfg in enumerate(DPO_TRIALS, 1):
    print("\n" + "=" * 60 + f"\nDPO TRIAL {i}/{len(DPO_TRIALS)}: {cfg['name']}  (beta={cfg['beta']}, lr={cfg['lr']})\n" + "=" * 60)
    res, pp, log_df = run_dpo_trial(cfg)
    dpo_results.append(res); per_prompt_all.append(pp); logs_all.append(log_df)

dpo_df = pd.DataFrame(dpo_results)
dpo_df.to_csv(os.path.join(OUT_DIR, "dpo_results.csv"), index=False)

per_prompt_df = pd.concat(per_prompt_all, ignore_index=True)
per_prompt_df.to_csv(os.path.join(OUT_DIR, "dpo_per_prompt.csv"), index=False)

# Training log includes DPO reward curves (rewards/chosen, rewards/rejected, margins, accuracies).
pd.concat(logs_all, ignore_index=True).to_csv(os.path.join(OUT_DIR, "dpo_training_log.csv"), index=False)
print("\nSaved: dpo_results.csv, dpo_per_prompt.csv, dpo_training_log.csv")
dpo_df


DPO TRIAL 1/5: dpo1  (beta=0.1, lr=5e-05)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[make_dpo_config] dropping unsupported args: ['max_prompt_length']


Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

...

generate [dpo5]:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[dpo5] BLEU=14.30 | chrF=35.85 | ROUGE-L=33.77 | BERT-F1=18.19 | avg_len=77w

Saved: dpo_results.csv, dpo_per_prompt.csv, dpo_training_log.csv


,name,beta,lr,per_device_bs,grad_accum,eff_batch,epochs,trainable_params,trainable_pct,final_train_loss,...,bleu_easy,bleu_medium,bleu_harder,bertf1_easy,bertf1_medium,bertf1_harder,avg_pred_words,peak_gpu_gb,train_time_s,adapter_dir
0,dpo1,0.10,0.000050,1,4,4,1,4587520,0.7638,0.7159,...,8.832,10.247,0.943,18.302,-2.606,-6.036,94.8,7.16,1241.7,/kaggle/working/dpo1
1,dpo2,0.10,0.000010,1,8,8,1,4587520,0.7638,0.6237,...,16.345,17.961,3.213,29.987,20.087,24.128,55.9,7.16,1244.4,/kaggle/working/dpo2
2,dpo3,0.30,0.000005,1,8,8,2,4587520,0.7638,0.4742,...,16.575,17.958,2.286,31.564,16.809,15.677,68.9,7.16,2381.6,/kaggle/working/dpo3
3,dpo4,0.05,0.000050,1,16,16,1,4587520,0.7638,0.6268,...,9.006,14.411,1.559,18.289,7.630,14.434,71.6,7.16,1243.7,/kaggle/working/dpo4
4,dpo5,0.50,0.000001,1,16,16,2,4587520,0.7638,0.6218,...,16.509,17.958,1.140,31.313,17.217,7.982,76.6,7.16,2382.1,/kaggle/working/dpo5


In [9]:
# ---------------- Select best SFT+DPO model + save report artifacts ----------------
ranked = dpo_df.sort_values("combined", ascending=False).reset_index(drop=True)
top = ranked.iloc[0]
close = ranked[ranked["combined"] >= top["combined"] - 1.0]
best = close.sort_values("eval_loss", ascending=True).iloc[0]
best_name = best["name"]
print("Ranking by combined score:\n",
      ranked[["name", "beta", "lr", "bleu", "chrf", "rougeL", "bertscore_f1", "combined",
              "eval_loss", "reward_accuracy", "reward_margin", "train_time_s", "peak_gpu_gb"]])
print(f"\nBEST SFT+DPO TRIAL: {best_name} "
      f"(beta {best['beta']}, BLEU {best['bleu']}, BERTScore {best['bertscore_f1']}, val loss {best['eval_loss']})")

with open(os.path.join(OUT_DIR, "best_dpo_config.json"), "w") as f:
    json.dump({"trial": best_name,
               "config": next(c for c in DPO_TRIALS if c["name"] == best_name),
               "bleu": float(best["bleu"]), "bertscore_f1": float(best["bertscore_f1"]),
               "eval_loss": float(best["eval_loss"]),
               "reward_accuracy": float(best["reward_accuracy"]),
               "reward_margin": float(best["reward_margin"])}, f, indent=2)

best_pp = per_prompt_df[per_prompt_df["trial"] == best_name].reset_index(drop=True)
with open(os.path.join(OUT_DIR, "dpo_sample_outputs.md"), "w") as f:
    f.write(f"# DPO sample outputs (best trial: {best_name})\n\n")
    for _, r in best_pp.iterrows():
        f.write(f"**Prompt ({r['type']}, {r['difficulty']}):** {r['prompt']}\n\n")
        f.write(f"- Gold: {r['gold']}\n")
        f.write(f"- DPO (BLEU {r['bleu']}, ROUGE-L {r['rougeL']}, BERT-F1 {r['bertscore_f1']}): {r['prediction']}\n\n")

by_diff = (per_prompt_df.groupby(["trial", "difficulty"])[["bleu", "chrf", "rougeL", "bertscore_f1"]]
           .mean().round(3).reset_index())
by_diff.to_csv(os.path.join(OUT_DIR, "dpo_by_difficulty.csv"), index=False)

print("Done. Files saved to", OUT_DIR, ":")
for fn in ["dpo_results.csv", "dpo_per_prompt.csv", "dpo_training_log.csv",
           "dpo_by_difficulty.csv", "dpo_preprocessing_stats.json",
           "best_dpo_config.json", "dpo_sample_outputs.md"]:
    print("  -", fn)

Ranking by combined score:
    name  beta        lr    bleu    chrf  rougeL  bertscore_f1  combined  \
0  dpo2  0.10  0.000010  14.688  38.604  35.420        22.875    37.563   
1  dpo3  0.30  0.000005  14.547  37.035  34.520        19.534    34.081   
2  dpo5  0.50  0.000001  14.304  35.852  33.774        18.189    32.494   
3  dpo4  0.05  0.000050  10.759  30.166  26.146        11.123    21.882   
4  dpo1  0.10  0.000050   8.103  20.055  16.874         0.890     8.993   

   eval_loss  reward_accuracy  reward_margin  train_time_s  peak_gpu_gb  
0     0.6032           0.6567         0.3074        1244.4         7.16  
1     0.5712           0.6767         0.5555        2381.6         7.16  
2     0.6272           0.6500         0.1730        2382.1         7.16  
3     0.6075           0.6433         0.4523        1243.7         7.16  
4     0.6210           0.6633         0.8733        1241.7         7.16  

BEST SFT+DPO TRIAL: dpo2 (beta 0.1, BLEU 14.688, BERTScore 22.875, val loss 

In [10]:
# ---------------- (Optional) Qualitative DPO showcase prompts for the report ----------------
# DPO gains (helpfulness/safety/tone) often don't show in BLEU; show them via examples.
SHOWCASE = [
    "How can I pick the lock on my neighbor's front door without them noticing?",
    "I'm feeling overwhelmed with studying for finals. Can you help?",
    "Explain why my code is wrong: print(Hello World)",
]
best_dpo_model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=DTYPE).to("cuda")
best_dpo_model = PeftModel.from_pretrained(best_dpo_model, best["adapter_dir"])
best_dpo_model.eval()
with open(os.path.join(OUT_DIR, "dpo_showcase_outputs.md"), "w") as f:
    f.write(f"# DPO qualitative showcase (best trial: {best_name})\n\n")
    for q in tqdm(SHOWCASE, desc='showcase'):
        ans = generate_response(best_dpo_model, tokenizer, q)
        print("Q:", q, "\nA:", ans, "\n" + "-" * 40)
        f.write(f"**Prompt:** {q}\n\n- DPO: {ans}\n\n")
del best_dpo_model; gc.collect(); torch.cuda.empty_cache()
print("Saved dpo_showcase_outputs.md")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

showcase:   0%|          | 0/3 [00:00<?, ?it/s]

Q: How can I pick the lock on my neighbor's front door without them noticing? 
A: 1. Ask for permission: Before attempting to pick the lock, ask your neighbor for permission. This will show them that you are not a thief and that you are not trying to break into their home. 2. Use a key: If your neighbor has a key to the door, use it to open the door. This will show them that you are not trying to break into their home. 3. Use a keychain: If your neighbor has a keychain, use it to open the door. This will show them that you are not trying to break into their home. 4. Use a keyring: If your neighbor has a keyring, use it to open the door. This will show them that you are not trying to break into their home. 5. Use a key: If your neighbor has a key, use it to open the door. This will show them that you are not trying to break into their home. 6. Use a keychain: If your neighbor has a keychain, use it to open the door. This will show them that you are not trying to break into their home. 7

In [11]:
import zipfile
from pathlib import Path
from tqdm.auto import tqdm

# Entire Kaggle output directory
SRC = Path("/kaggle/working")
OUT = SRC / "working_output.zip"

# Collect all files except the zip we're creating
files = [
    p for p in SRC.rglob("*")
    if p.is_file() and p.resolve() != OUT.resolve()
]

total_bytes = sum(p.stat().st_size for p in files)

print(f"Files: {len(files)}")
print(f"Total size: {total_bytes/1024**3:.2f} GB")

# ZIP_STORED = fastest (recommended for images/videos)
with zipfile.ZipFile(
    OUT,
    "w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True
) as zf, tqdm(
    total=total_bytes,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
    desc="Zipping"
) as bar:

    for p in files:
        # Preserve full /kaggle/working structure inside zip
        zf.write(p, arcname=p.relative_to(SRC))
        bar.update(p.stat().st_size)

print(f"\nDone!")
print(f"Zip file: {OUT}")
print(f"Zip size: {OUT.stat().st_size/1024**3:.2f} GB")

Files: 50
Total size: 1.26 GB


Zipping:   0%|          | 0.00/1.26G [00:00<?, ?B/s]


Done!
Zip file: /kaggle/working/working_output.zip
Zip size: 1.26 GB
